<a href="https://colab.research.google.com/github/VamshiPoojari/Finance_Club_open_project_2026/blob/main/Vamshi_Poojari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

class YieldCurvePreprocessor:
    def __init__(self, file_path):

        self.file_path = file_path
        self.df = None

    def load_and_inspect(self):

        # Load the raw data
        self.df = pd.read_csv(self.file_path)

        print("--- Dataset Info ---")
        # This will reveal formatting inconsistencies (e.g., if a numeric column is loaded as an 'object' or string)
        self.df.info()

        print("\n--- First 5 Rows ---")
        display(self.df.head())

        print("\n--- Missing Values Count ---")
        # Identifies columns with nulls that we'll need to forward-fill or interpolate later
        print(self.df.isnull().sum())

        print("\n--- Statistical Summary ---")
        # Helps spot extreme outliers (e.g., negative yields or massive spikes)
        display(self.df.describe())

        return self.df

# --- Execution Block ---

preprocessor = YieldCurvePreprocessor('train_data.csv')
raw_df = preprocessor.load_and_inspect()

### Data Engineering and Preprocessing: Initial Inspection

Before calibrating the **Cox-Ingersoll-Ross (CIR) model**, a robust inspection of the historical dataset containing daily bond yields is a mandatory first step. The dataset provides yields across 9 specific maturity tenors, ranging from 3 Months to 30 Years.  

Based on the initial data ingestion and statistical summary, we can observe the following critical characteristics necessary to ensure the data is mathematically viable for time-series calibration:  

* **Data Shape & Completeness:** The training dataset consists of 1,976 daily observations. An initial check indicates no explicitly missing (null) values in this raw format.
* **Formatting Inconsistencies:** The column names contain leading spaces (e.g., `' ZC025YR'`). These string formatting anomalies must be cleaned and normalized to prevent key-errors during matrix operations and curve reconstruction.  
* **Positivity Constraint (Mathematical Viability):** The CIR framework relies on a mean-reverting square-root diffusion process, which requires interest rates to remain strictly positive. The statistical summary confirms that the minimum yields across all tenors are strictly greater than zero (ranging from a minimum of **0.04%** at the short end to maximums near **5.4%**), satisfying a core macroeconomic baseline for the base model.

In [ ]:
import pandas as pd
import numpy as np

class YieldCurvePreprocessor:
    def __init__(self, file_path):

        self.file_path = file_path
        self.df = None

    def load_and_inspect(self):

        self.df = pd.read_csv(self.file_path)
        return self.df

    def plot_and_verify(self):
        """
        Plots the time series of the yields to visually inspect for anomalies.
        Verifies that all rates are strictly positive (a requirement for the CIR model).
        """
        import matplotlib.pyplot as plt

        plt.figure(figsize=(14, 7))

        # Plotting all tenors.
        # Using a colormap makes it easier to distinguish short-term vs long-term rates.
        colors = plt.cm.viridis(np.linspace(0, 1, len(self.df.columns)))

        for i, col in enumerate(self.df.columns):
            plt.plot(self.df.index, self.df[col], label=col, color=colors[i], alpha=0.8, linewidth=1.2)

        plt.title('Historical Yield Curve Evolution', fontsize=14, fontweight='bold')
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Yield (Decimal)', fontsize=12)

        # Place legend outside the plot
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Maturities")
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        # --- Mathematical Sanity Check ---
        min_yield = self.df.min().min()
        max_yield = self.df.max().max()

        print("\n--- CIR Mathematical Sanity Check ---")
        print(f"Absolute Minimum Yield: {min_yield:.6f}")
        print(f"Absolute Maximum Yield: {max_yield:.6f}")

        if min_yield <= 0:
            print("❌ WARNING: Negative or zero yields detected! The base CIR model will fail (requires strictly positive rates).")
        else:
            print("✅ All yields are strictly positive. The dataset is mathematically viable for the base CIR model.")

    def clean_data(self):

        # 1. Strip hidden whitespaces from column names
        self.df.columns = self.df.columns.str.strip()

        # 2. Rename columns to the human-readable tenors
        tenor_mapping = {
            'ZC025YR': '3M', 'ZC050YR': '6M', 'ZC075YR': '9M',
            'ZC100YR': '1Y', 'ZC200YR': '2Y', 'ZC500YR': '5Y',
            'ZC1000YR': '10Y', 'ZC2000YR': '20Y', 'ZC3000YR': '30Y'
        }
        self.df.rename(columns=tenor_mapping, inplace=True)

        # 3. Convert 'Date' to datetime and set it as the index
        self.df['Date'] = pd.to_datetime(self.df['Date'])
        self.df.set_index('Date', inplace=True)
        self.df.sort_index(inplace=True)

        # 4. Enforce a Business Day (B) frequency to expose missing non-trading days
        full_b_days = pd.date_range(start=self.df.index.min(), end=self.df.index.max(), freq='B')
        missing_days_count = len(full_b_days) - len(self.df)
        print(f"--- Enforcing Calendar ---")
        print(f"Exposed {missing_days_count} missing business days (e.g., holidays).")

        # Reindex to the full business calendar. This introduces NaNs for missing days.
        self.df = self.df.reindex(full_b_days)

        # 5. Handle missing values via forward-filling
        # (Forward-filling assumes the yield on a holiday remains the same as the previous trading day)
        self.df.ffill(inplace=True)

        print("\n--- Data Cleaning Complete ---")
        print(f"Total rows after calendar enforcement: {len(self.df)}")
        print(f"Remaining Missing Values: {self.df.isnull().sum().sum()}")

        return self.df

# Create fresh instance and run the pipeline
preprocessor = YieldCurvePreprocessor('train_data.csv')
preprocessor.load_and_inspect()
cleaned_df = preprocessor.clean_data()

# Run our new visualization and check
preprocessor.plot_and_verify()

### Data Engineering and Preprocessing: Cleaning and Validation

To prepare the data for time-series calibration, we executed a robust preprocessing pipeline to handle formatting inconsistencies and non-trading day anomalies.  

**Key Preprocessing Steps:**

* **Standardization:** Hidden whitespaces in column headers were stripped, and variables were renamed to standard maturity tenors (3M through 30Y) for clean programmatic access.
* **Calendar Enforcement & Imputation:** Financial time-series data often skips holidays and weekends. By enforcing a strict Business Day (B) frequency, we exposed 96 missing trading days. These were resolved using forward-filling, a standard practice that assumes the yield remains constant from the previous trading day, preventing look-ahead bias.  
* **Yield Curve Evolution:** The generated plot visualizes the historical term structure. We can observe distinct market regimes, including periods of rate compression (where short and long-term yields converge) and significant macroeconomic shocks (such as the sharp drop in rates around early 2020).
* **CIR Mathematical Sanity Check:** The fundamental feature of the Cox-Ingersoll-Ross (CIR) model is its mean-reverting square-root diffusion process. The stochastic differential equation for the short rate $r_t$ is defined as:  

$$dr_t = \kappa(\theta - r_t)dt + \sigma\sqrt{r_t}dW_t$$
  
For the term $\sigma\sqrt{r_t}$ to remain a real number, the instantaneous rate $r_t$ must never fall below zero. Our sanity check confirms that the absolute minimum yield in the dataset is 0.000486 (approx. **0.048%**). Because all rates are strictly positive, the dataset satisfies this crucial mathematical constraint and is viable for the base CIR implementation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize

class CIRCalibrator:
    def __init__(self, dt=1/252):

        self.dt = dt
        self.kappa = None
        self.theta = None
        self.sigma = None


    def calibrate_ols(self, rates_series):
        rates = rates_series.values
        r_t = rates[:-1]
        r_t_plus_1 = rates[1:]

        r_t_safe = np.maximum(r_t, 1e-8)

        # Standardize matrix for homoscedasticity
        Y = (r_t_plus_1 - r_t) / np.sqrt(r_t_safe)
        X1 = 1.0 / np.sqrt(r_t_safe)
        X2 = np.sqrt(r_t_safe)

        X = np.column_stack((X1, X2))

        model = LinearRegression(fit_intercept=False)
        model.fit(X, Y)

        beta1, beta2 = model.coef_

        self.kappa = -beta2 / self.dt
        self.theta = beta1 / (self.kappa * self.dt)

        predictions = model.predict(X)
        residuals = Y - predictions
        self.sigma = np.std(residuals) / np.sqrt(self.dt)

        print("--- OLS Calibration Successful ---")
        self._print_results()

        return self.kappa, self.theta, self.sigma



    def calibrate_mle(self, rates_series):
        """
        Calibrates the CIR model using Maximum Likelihood Estimation (MLE).
        Automatically handles scaling by 100 internally to prevent optimizer failure
        on tiny decimal values, and mathematically unscales the results.
        """
        scale_factor = 100.0
        rates = rates_series.values * scale_factor

        r_t = rates[:-1]
        r_t_plus_1 = rates[1:]
        dr = r_t_plus_1 - r_t
        r_t_safe = np.maximum(r_t, 1e-8)

        def negative_log_likelihood(params):
            k, th, sig = params
            drift = k * (th - r_t_safe) * self.dt
            variance = (sig ** 2) * r_t_safe * self.dt

            # Prevent log(0) or log(negative) in variance
            variance = np.maximum(variance, 1e-10)

            nll = np.sum(0.5 * np.log(2 * np.pi * variance) + ((dr - drift) ** 2) / (2 * variance))
            return nll

        # Initial guesses adjusted for percentages (e.g., theta = 2.0%)
        initial_guess = [0.1, 2.5, 0.5]

        # Bounds for scaled data: kappa(0 to 5), theta(0 to 20%), sigma(0 to 5)
        bounds = ((1e-4, 5.0), (1e-4, 20.0), (1e-4, 5.0))

        result = minimize(negative_log_likelihood, initial_guess, bounds=bounds, method='L-BFGS-B')

        if not result.success:
            print("⚠️ Optimizer failed to converge. Results may be unstable.")

        kappa_opt, theta_opt, sigma_opt = result.x

        self.kappa = kappa_opt
        self.theta = theta_opt / scale_factor
        self.sigma = sigma_opt / np.sqrt(scale_factor) # Volatility scales by sqrt(c)

        print("--- MLE Calibration Successful ---")
        self._print_results()

        return self.kappa, self.theta, self.sigma


    def calibrate_gmm(self, rates_series):
        """
        Calibrates the CIR model using the Generalised Method of Moments (GMM).
        Uses four empirical moment conditions derived from Euler discretization.
        """
        scale_factor = 100.0
        rates = rates_series.values * scale_factor

        r_t = rates[:-1]
        r_t_plus_1 = rates[1:]
        r_t_safe = np.maximum(r_t, 1e-8)

        def gmm_objective(params):
            k, th, sig = params

            # 1. Discretization Errors
            drift_error = (r_t_plus_1 - r_t) - k * (th - r_t_safe) * self.dt
            variance_error = (drift_error ** 2) - (sig ** 2) * r_t_safe * self.dt

            # 2. Define Sample Moments (Expectation should be 0)
            m1 = np.mean(drift_error)
            m2 = np.mean(drift_error * r_t_safe)
            m3 = np.mean(variance_error)
            m4 = np.mean(variance_error * r_t_safe)

            # 3. Objective is to minimize the sum of squared moments (Identity Weight Matrix)
            return m1**2 + m2**2 + m3**2 + m4**2

        initial_guess = [0.1, 2.5, 0.5]
        bounds = ((1e-4, 5.0), (1e-4, 20.0), (1e-4, 5.0))

        result = minimize(gmm_objective, initial_guess, bounds=bounds, method='L-BFGS-B')

        if not result.success:
            print("⚠️ GMM Optimizer failed to converge.")

        kappa_opt, theta_opt, sigma_opt = result.x

        self.kappa = kappa_opt
        self.theta = theta_opt / scale_factor
        self.sigma = sigma_opt / np.sqrt(scale_factor)

        print("--- GMM Calibration Successful ---")
        self._print_results()

        return self.kappa, self.theta, self.sigma


    def calibrate_kalman(self, rates_series):
        """
        Calibrates the CIR model using a 1D Extended Kalman Filter (EKF) structure.
        Treats the CIR process as a hidden state and the observed yields as noisy measurements.
        Minimizes the negative log-likelihood of the prediction errors.
        """
        scale_factor = 100.0
        measurements = rates_series.values * scale_factor

        def kalman_log_likelihood(params):
            k, th, sig, R_noise = params

            n = len(measurements)
            log_lik = 0.0

            # Initial state estimates
            r_hat = measurements[0]  # initial state
            P = 1e-4                 # initial uncertainty

            for i in range(1, n):
                z = measurements[i]

                # 1. Predict Step
                r_pred = r_hat + k * (th - r_hat) * self.dt
                r_pred = max(r_pred, 1e-8) # Enforce positivity

                # State-dependent variance approximation (EKF style)
                Q = (sig ** 2) * r_pred * self.dt
                P_pred = P * ((1 - k * self.dt) ** 2) + Q

                # 2. Measurement Update Step
                y = z - r_pred           # Innovation (prediction error)
                S = P_pred + R_noise     # Innovation covariance

                # Log-likelihood contribution of this step
                S_safe = max(S, 1e-10)
                log_lik -= 0.5 * (np.log(2 * np.pi * S_safe) + (y ** 2) / S_safe)

                # 3. Update State
                K = P_pred / S_safe      # Kalman Gain
                r_hat = r_pred + K * y
                P = (1 - K) * P_pred

            return -log_lik # We minimize negative log-likelihood

        # Initial guess includes measurement noise R: [kappa, theta, sigma, R_noise]
        initial_guess = [0.1, 2.5, 0.5, 0.01]
        bounds = ((1e-4, 5.0), (1e-4, 20.0), (1e-4, 5.0), (1e-6, 1.0))

        result = minimize(kalman_log_likelihood, initial_guess, bounds=bounds, method='L-BFGS-B')

        if not result.success:
            print("⚠️ Kalman Filter optimizer failed to converge.")

        kappa_opt, theta_opt, sigma_opt, R_opt = result.x

        self.kappa = kappa_opt
        self.theta = theta_opt / scale_factor
        self.sigma = sigma_opt / np.sqrt(scale_factor)

        print("--- Kalman Filter Calibration Successful ---")
        print(f"Estimated Measurement Noise (R): {(R_opt / scale_factor):.6f}")
        self._print_results()

        return self.kappa, self.theta, self.sigma


    def _print_results(self):
        """Helper to print parameters and check Feller condition."""
        print(f"Speed of mean reversion (kappa): {self.kappa:.4f}")
        print(f"Long-run mean (theta):           {self.theta:.4f}")
        print(f"Volatility coefficient (sigma):  {self.sigma:.4f}")

        feller_val = 2 * self.kappa * self.theta
        sigma_sq = self.sigma ** 2

        print("\n--- Feller Condition Check ---")
        if feller_val >= sigma_sq:
            print(f"✅ PASSED: 2*kappa*theta ({feller_val:.6f}) >= sigma^2 ({sigma_sq:.6f})")
            print("Rates are mathematically guaranteed to remain strictly positive.\n")
        else:
            print(f" FAILED: 2*kappa*theta ({feller_val:.6f}) < sigma^2 ({sigma_sq:.6f})")
            print("Note: In real-world data, Feller often fails due to high short-term volatility.\n")


# --- Execution Example ---

calibrator = CIRCalibrator(dt=1/252)
short_rate_series = cleaned_df['3M']

print("\n================= OLS =================")
calibrator.calibrate_ols(short_rate_series)

print("\n================= MLE =================")
calibrator.calibrate_mle(short_rate_series)

print("\n================= GMM =================")
calibrator.calibrate_gmm(short_rate_series)

print("\n================ KALMAN ===============")
calibrator.calibrate_kalman(short_rate_series)

### Base CIR Model Calibration: Mathematical Analysis

The Cox-Ingersoll-Ross (CIR) model relies on the stochastic differential equation:

$$dr_t = \kappa(\theta - r_t)dt + \sigma\sqrt{r_t}dW_t$$

To extract the parameters $(\kappa, \theta, \sigma)$ from discrete historical data, we must discretize this continuous-time process. The most common approach is the Euler-Maruyama discretization:

$$r_{t+1} - r_t = \kappa(\theta - r_t)\Delta t + \sigma\sqrt{r_t}\epsilon_t$$

where $\epsilon_t \sim \mathcal{N}(0, \Delta t)$. Our code explores four distinct methodologies to solve for these parameters.

**1. Ordinary Least Squares (OLS)**
To use linear regression, we divide the discretized equation by $\sqrt{r_t}$ to eliminate heteroscedasticity (ensuring the error term variance is constant):

$$\frac{r_{t+1} - r_t}{\sqrt{r_t}} = \frac{\kappa\theta\Delta t}{\sqrt{r_t}} - \kappa\sqrt{r_t}\Delta t + \sigma\epsilon_t$$

This perfectly maps to a linear regression model without an intercept, $Y = \beta_1X_1 + \beta_2X_2 + \epsilon$, where:
* $Y = \frac{r_{t+1} - r_t}{\sqrt{r_t}}$
* $X_1 = \frac{1}{\sqrt{r_t}}$ (implies $\beta_1 = \kappa\theta\Delta t$)
* $X_2 = \sqrt{r_t}$ (implies $\beta_2 = -\kappa\Delta t$)

**2. Maximum Likelihood Estimation (MLE)**
While the true transition density of the CIR model is a non-central chi-squared distribution, for high-frequency (daily) data, we can approximate the transition probabilities using a Gaussian distribution. The log-likelihood function minimizes the prediction errors relative to the state-dependent variance:

$$\mathcal{L}(\kappa, \theta, \sigma) = \sum_{t} \left( -\frac{1}{2}\ln(2\pi\sigma^2 r_t \Delta t) - \frac{(r_{t+1} - r_t - \kappa(\theta - r_t)\Delta t)^2}{2\sigma^2 r_t \Delta t} \right)$$

The code minimizes the negative of this sum.

**3. Generalised Method of Moments (GMM)**
GMM avoids assuming a specific distribution (like the Gaussian approximation in MLE). Instead, it relies on sample moments derived from the discretization errors matching theoretical expectations (which should be zero).

The code defines the drift error $e_t = r_{t+1} - r_t - \kappa(\theta - r_t)\Delta t$ and variance error $v_t = e_t^2 - \sigma^2 r_t \Delta t$. It then minimizes the sum of squared moments:

$$m_1 = \mathbb{E}[e_t] = 0$$
$$m_2 = \mathbb{E}[e_t r_t] = 0$$
$$m_3 = \mathbb{E}[v_t] = 0$$
$$m_4 = \mathbb{E}[v_t r_t] = 0$$

**4. Extended Kalman Filter (EKF)**
The Kalman Filter treats the true interest rate as a "hidden state" and the observed yields as noisy measurements.
* **Predict Step:** Projects the rate forward using the deterministic drift: $r_{pred} = r_{hat} + \kappa(\theta - r_{hat})\Delta t$.
* **Update Step:** Corrects the prediction using the actual observed rate, weighted by the "Kalman Gain" ($K$), which balances our trust in the model's prediction versus the noisy observation.

---

### Model Mechanics and Calibration: Results Interpretation

Based on the output, the calibrated yield curve is highly sensitive to the choice of calibration methodology. We have two explicit failures and two passes, highlighting the difficulty of fitting theoretical models to noisy market data.

#### The Two Failures: OLS and GMM

* **OLS (Catastrophic Theoretical Failure):** While the console output prints **✅ PASSED** for the Feller condition, OLS is fundamentally incorrect here. It estimated a negative mean reversion ($\kappa = -0.2422$) and a negative long-run mean ($\theta = -0.0053$). In the CIR framework, $\kappa < 0$ implies "mean aversion" (rates will explode away from the mean rather than pull back), and a negative $\theta$ violates the premise of positive interest rates. The Feller check ($2\kappa\theta \ge \sigma^2$) only mathematically passed because multiplying two negative numbers resulted in a positive product.
* **GMM (Feller Condition Failure):** GMM produced economically plausible parameters ($\theta \approx 2.51\%$), but it explicitly failed the Feller condition check. The estimated mean reversion was incredibly weak ($\kappa = 0.0001$), meaning the short-term volatility ($\sigma^2$) overpowered the drift. Under these parameters, the model cannot mathematically guarantee that rates will remain strictly positive during simulations.

#### The Two Passes: MLE and Kalman Filter

* **MLE (Mathematical Pass, Practical Warning):** MLE successfully converged and strictly passed the Feller condition. However, it estimated a long-run mean ($\theta$) of **20.00%**. This is a common artifact when MLE algorithms encounter highly persistent time-series data with extreme short-term shocks (like 2020); the optimizer pushes the long-run mean to its absolute upper bound in an attempt to explain the variance. While it passes mathematical checks, a 20% long-run 3M yield is macroeconomically unrealistic for this dataset.
* **Kalman Filter (The Robust Pass):** The Extended Kalman Filter provides the most balanced and realistic calibration. It passes the Feller condition with a realistic mean-reversion speed ($\kappa = 0.0750$) and a highly plausible long-run mean ($\theta = 3.25\%$). By separating the "true" state of the interest rate from the "measurement noise" of the market, the Kalman Filter prevented short-term volatility spikes from permanently distorting the fundamental structural parameters.

**Implications of Mean-Reversion ($\kappa$):**
In the successful Kalman model, $\kappa = 0.0750$ implies a relatively slow persistence of interest rate shocks. When a macroeconomic shock pushes the 3M rate away from its **3.25%** baseline, the pull-back is gradual, reflecting the slow-moving nature of central bank policy adjustments in real-world data.

In [ ]:
class CIRYieldCurveBuilder:
    def __init__(self, kappa, theta, sigma):
        """
        Initializes the builder with our calibrated MLE parameters.
        """
        self.kappa = kappa
        self.theta = theta
        self.sigma = sigma

    def _compute_A_B(self, tau):
        """
        Internal method to calculate the deterministic functions A(t,T) and B(t,T).
        tau is the time to maturity in years.
        """
        # gamma = sqrt(kappa^2 + 2*sigma^2)
        gamma = np.sqrt(self.kappa**2 + 2 * self.sigma**2)

        # Calculate B(t,T)
        exp_gamma_tau = np.exp(gamma * tau)
        denominator = (gamma + self.kappa) * (exp_gamma_tau - 1) + 2 * gamma

        B = (2 * (exp_gamma_tau - 1)) / denominator

        # Calculate A(t,T)
        numerator_A = 2 * gamma * np.exp((self.kappa + gamma) * tau / 2)
        A = (numerator_A / denominator) ** ((2 * self.kappa * self.theta) / self.sigma**2)

        return A, B

    def predict_yields(self, r_t, tenors_years):
        """
        Reconstructs the yield curve for given maturities based purely on the short rate r_t.
        """
        predicted_yields = []
        for tau in tenors_years:
            A, B = self._compute_A_B(tau)

            # The closed-form yield formula
            y = (B * r_t - np.log(A)) / tau
            predicted_yields.append(y)

        return np.array(predicted_yields)

# --- Execution Block ---

# 1. Explicitly capture the Kalman Filter parameters
kappa_kf, theta_kf, sigma_kf = calibrator.calibrate_kalman(short_rate_series)

print("\n--- Initializing Yield Curve Builder ---")
# 2. Instantiate the builder with the Kalman parameters
builder = CIRYieldCurveBuilder(kappa=kappa_kf, theta=theta_kf, sigma=sigma_kf)

# 3. Define the maturities in years corresponding to our test columns
# (6M, 9M, 1Y, 2Y, 5Y, 10Y, 20Y, 30Y)
tenors_years = [0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0]
tenor_labels = ['6M', '9M', '1Y', '2Y', '5Y', '10Y', '20Y', '30Y']

# 4. Let's test the prediction on the very first day of your cleaned training dataset
first_day_3M_rate = cleaned_df['3M'].iloc[0]
actual_yields = cleaned_df.iloc[0, 1:].values # Grab actuals for comparison (excluding 3M)

# 5. Generate the prediction
predicted_yields = builder.predict_yields(first_day_3M_rate, tenors_years)

print(f"\nInput 3M Rate (r_t): {first_day_3M_rate:.6f}\n")
print(f"{'Tenor':<10} | {'Predicted Yield':<20} | {'Actual Yield':<20}")
print("-" * 55)
for label, pred, actual in zip(tenor_labels, predicted_yields, actual_yields):
    print(f"{label:<10} | {pred:<20.6f} | {actual:<20.6f}")




# --- Execution Block: MLE & Kalman Separation ---

# 1. Explicitly capture parameters for BOTH models to avoid variable collision
# (Assuming 'calibrator' and 'short_rate_series' are still in memory)
print("--- Calibrating MLE ---")
kappa_mle, theta_mle, sigma_mle = calibrator.calibrate_mle(short_rate_series)

print("\n--- Calibrating Kalman Filter ---")
kappa_kf, theta_kf, sigma_kf = calibrator.calibrate_kalman(short_rate_series)

print("\n--- Initializing Distinct Yield Curve Builders ---")
# 2. Instantiate TWO distinct builders
mle_builder = CIRYieldCurveBuilder(kappa=kappa_mle, theta=theta_mle, sigma=sigma_mle)
kf_builder = CIRYieldCurveBuilder(kappa=kappa_kf, theta=theta_kf, sigma=sigma_kf)

# 3. Define the maturities in years corresponding to our test columns
tenors_years = [0.5, 0.75, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0]
tenor_labels = ['6M', '9M', '1Y', '2Y', '5Y', '10Y', '20Y', '30Y']

# 4. Grab the first day's data for our sanity check
first_day_3M_rate = cleaned_df['3M'].iloc[0]
actual_yields = cleaned_df.iloc[0, 1:].values # Grab actuals for comparison (excluding 3M)

# 5. Generate predictions from BOTH builders separately
mle_predicted_yields = mle_builder.predict_yields(first_day_3M_rate, tenors_years)
kf_predicted_yields = kf_builder.predict_yields(first_day_3M_rate, tenors_years)

# 6. Output a side-by-side comparative table
print(f"\nInput 3M Rate (r_t): {first_day_3M_rate:.6f}\n")
print(f"{'Tenor':<6} | {'Actual Yield':<15} | {'MLE Prediction':<15} | {'KF Prediction':<15}")
print("-" * 65)
for label, actual, mle_pred, kf_pred in zip(tenor_labels, actual_yields, mle_predicted_yields, kf_predicted_yields):
    print(f"{label:<6} | {actual:<15.6f} | {mle_pred:<15.6f} | {kf_pred:<15.6f}")

### The Prediction Challenge: Yield Curve Construction

Having calibrated the structural parameters $(\kappa, \theta, \sigma)$, the next step is to reconstruct the entire yield curve (6M to 30Y) using only the 3-Month rate as the instantaneous short rate proxy ($r_t$).

**Mathematical Framework:**
The CIR model provides an affine term structure, meaning the yield of a zero-coupon bond is a linear function of the short rate. The continuously compounded yield $y(t,\tau)$ for a given maturity $\tau$ is derived from the bond price $P(t,T)$:

$$y(t,\tau) = \frac{B(t,\tau)r_t - \ln A(t,\tau)}{\tau}$$

Where the deterministic functions $A(t,\tau)$ and $B(t,\tau)$ are calculated via the Riccati equations. Let $\gamma = \sqrt{\kappa^2 + 2\sigma^2}$:

$$B(t,\tau) = \frac{2(e^{\gamma\tau} - 1)}{(\gamma + \kappa)(e^{\gamma\tau} - 1) + 2\gamma}$$

$$A(t,\tau) = \left( \frac{2\gamma e^{(\kappa + \gamma)\tau/2}}{(\gamma + \kappa)(e^{\gamma\tau} - 1) + 2\gamma} \right)^{\frac{2\kappa\theta}{\sigma^2}}$$

Our `CIRYieldCurveBuilder` class implements these exact closed-form solutions to generate the theoretical yields.

---

### Prediction and Model Performance: Initial Calibration Comparison

By testing our models on the first day of the dataset ($r_t =$ **0.528%**), we can immediately observe the practical implications of our previous calibration choices (MLE vs. Kalman Filter) and identify where the single-factor CIR model struggles.

**1. The Impact of Parameter Skew (MLE vs. KF):**
* **MLE's Long-End Overshoot:** As noted in the calibration analysis, the MLE optimizer pushed the long-run mean ($\theta$) to an unrealistic **20.0%**. This mathematical artifact heavily distorts the reconstructed curve. While MLE fits the short end (6M, 9M) relatively well, it drastically overestimates the long end (predicting a 30Y yield of **3.11%** against an actual **2.04%**). The high $\theta$ forces the model to steepen the curve excessively.
* **Kalman Filter's Balanced Fit:** The Kalman Filter, with its more realistic $\theta =$ **3.25%**, performs significantly better across the entire term structure. Its 30Y prediction is **2.07%**, nearly perfectly matching the actual **2.04%**.

**2. Identifying Structural Limitations (Where the Base Model Fails):**
Even with the superior Kalman calibration, we can see systematic mispricings inherent to a single-factor model:
* **The Belly of the Curve:** The 2Y and 5Y tenors are the hardest to fit accurately. For example, KF predicts the 5Y at **0.97%**, while the actual is **0.79%**.
* **Why does this happen?** The base CIR model relies entirely on a single stochastic factor ($r_t$). Because it only knows the current 3M rate and the long-run mean, it restricts the yield curve to three primary shapes: monotonically increasing, monotonically decreasing, or slightly humped. It cannot capture complex "kinks" or independent movements in the medium-term maturities (often driven by 2-to-5-year central bank policy expectations rather than the instantaneous overnight rate).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

master_tenor_map = {
    '6M': 0.5, '9M': 0.75, '1Y': 1.0, '2Y': 2.0,
    '5Y': 5.0, '10Y': 10.0, '20Y': 20.0, '30Y': 30.0
}
tenor_labels = ['6M', '9M', '1Y', '2Y', '5Y', '10Y', '20Y', '30Y']
tenors_to_predict_years = [master_tenor_map[label] for label in tenor_labels]

# 1. Instantiate BOTH builders (using the explicit variables we defined earlier)
mle_builder = CIRYieldCurveBuilder(kappa=kappa_mle, theta=theta_mle, sigma=sigma_mle)
kf_builder = CIRYieldCurveBuilder(kappa=kappa_kf, theta=theta_kf, sigma=sigma_kf)

# 2. Generate prediction DataFrames for both models
all_mle_yields, all_kf_yields = [], []

for r_t in cleaned_df['3M'].values:
    all_mle_yields.append(mle_builder.predict_yields(r_t, tenors_to_predict_years))
    all_kf_yields.append(kf_builder.predict_yields(r_t, tenors_to_predict_years))

predicted_df_mle = pd.DataFrame(all_mle_yields, index=cleaned_df.index, columns=tenor_labels)
predicted_df_kf = pd.DataFrame(all_kf_yields, index=cleaned_df.index, columns=tenor_labels)

# 3. Set up the visualization (2 rows, 1 column, shared X-axis)
fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(18, 16), sharex=True)
colors = plt.cm.viridis(np.linspace(0, 1, len(tenor_labels)))

# --- TOP PLOT: MLE (Cross-Sectional) ---
for tenor, color in zip(tenor_labels, colors):
    ax1.plot(cleaned_df.index, cleaned_df[tenor], color=color, alpha=0.4, linewidth=3, label=f'Actual {tenor}')
    ax1.plot(predicted_df_mle.index, predicted_df_mle[tenor], color=color, linestyle='--', linewidth=1.5, label=f'MLE {tenor}')

ax1.set_title('Base CIR Model Backtest: MLE (Cross-Sectional) Parameters', fontsize=16, fontweight='bold')
ax1.set_ylabel('Yield (Decimal)', fontsize=14)
ax1.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10, ncol=2)
ax1.grid(True, linestyle='--', alpha=0.5)

# --- BOTTOM PLOT: Kalman Filter (Time-Series) ---
for tenor, color in zip(tenor_labels, colors):
    ax2.plot(cleaned_df.index, cleaned_df[tenor], color=color, alpha=0.4, linewidth=3, label=f'Actual {tenor}')
    ax2.plot(predicted_df_kf.index, predicted_df_kf[tenor], color=color, linestyle='--', linewidth=1.5, label=f'Kalman {tenor}')

ax2.set_title('Base CIR Model Backtest: Kalman Filter Parameters', fontsize=16, fontweight='bold')
ax2.set_xlabel('Date', fontsize=14)
ax2.set_ylabel('Yield (Decimal)', fontsize=14)
ax2.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=10, ncol=2)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Prediction from the Graph

By backtesting our calibrated models across the entire historical timeline, the visual divergence between the actual yields (solid lines) and our model predictions (dashed lines) vividly illustrates both the impact of our calibration methodologies and the fundamental limitations of the single-factor CIR framework.

**1. The MLE Backtest: Systematic Overestimation**

Looking at the MLE (Cross-Sectional) Parameters plot, we observe a catastrophic breakdown in the model's predictive power for longer-dated maturities.
* **The Observation:** The dashed lines representing the 10Y, 20Y, and 30Y predictions (the yellow/light-green lines) "float" significantly higher than the actual market yields for almost the entire historical period.
* **The "Why":** As identified during the calibration phase, the MLE optimizer was confused by the extreme volatility in the raw data and pushed the long-run mean ($\theta$) to an artificial ceiling of **20.0%**. Because the CIR model mathematically pulls the yield curve toward this $\theta$ over time, the model systematically overestimates the long end of the curve, falsely projecting massive term premia that do not exist in reality.

**2. The Kalman Filter Backtest: The Single-Factor Rigidity**

The Kalman Filter Parameters plot is much more realistic, keeping the predictions anchored closer to reality thanks to a plausible $\theta$ (**3.25%**). However, it explicitly reveals the structural weaknesses of a single-factor short-rate model:
* **Over-reliance on the 3M Anchor:** The model reconstructs the curve using only the current 3-Month rate ($r_t$). Consequently, the predicted yield curves essentially look like parallel shifts or simple blends between the current 3M rate and the static **3.25%** long-run mean.
* **Failure to Capture Curvature:** Which maturities are hardest to fit? The "belly" of the curve (2Y to 10Y). In real markets, medium-term yields often move independently of the overnight rate due to changing central bank expectations or supply/demand dynamics (causing "humps" or "inversions"). Because the base CIR model only has one stochastic driver ($dW_t$), it completely misses these nuanced shape changes, resulting in periods where it systematically underestimates the 5Y while simultaneously overestimating the 30Y.

---

### Conclusion for the Base Model

How accurately can the 3M rate alone reconstruct the full yield curve? It cannot do so with high precision across distinct macroeconomic regimes. While the Kalman Filter proves that rigorous calibration can prevent the model from exploding, the base CIR model acts as a rigid "rubber band" stretched between $r_t$ and $\theta$. It completely fails to capture sudden market shocks where the slope of the curve inverts or steepens independently of the short rate. This visual evidence provides the exact mathematical justification needed to introduce a Two-Factor CIR model or Time-Dependent parameters (CIR++) in the next phase of the project.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from scipy.optimize import minimize
from scipy.stats import ncx2
from typing import Dict, Optional, Tuple

# ==============================================================================
# 1. YieldCurvePreprocessor
# ==============================================================================
class YieldCurvePreprocessor:
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.df = None
        self.tenor_mapping = {
            'ZC025YR': '3M', 'ZC050YR': '6M', 'ZC075YR': '9M',
            'ZC100YR': '1Y', 'ZC200YR': '2Y', 'ZC500YR': '5Y',
            'ZC1000YR': '10Y', 'ZC2000YR': '20Y', 'ZC3000YR': '30Y'
        }

    def process(self) -> pd.DataFrame:
        self.df = pd.read_csv(self.file_path)
        self.df.columns = self.df.columns.str.strip()
        self.df.rename(columns=self.tenor_mapping, inplace=True)
        self.df['Date'] = pd.to_datetime(self.df['Date'])
        self.df.set_index('Date', inplace=True)
        self.df.sort_index(inplace=True)
        full_b_days = pd.date_range(start=self.df.index.min(), end=self.df.index.max(), freq='B')
        self.df = self.df.reindex(full_b_days)
        self.df.ffill(inplace=True)
        return self.df

# ==============================================================================
# 2. CIRYieldCurveBuilder Class
# ==============================================================================
class CIRYieldCurveBuilder:
    def __init__(self, kappa, theta, sigma):
        self.kappa = kappa
        self.theta = theta
        self.sigma = sigma

    def _compute_A_B(self, tau):
        gamma = np.sqrt(self.kappa**2 + 2 * self.sigma**2)
        exp_gamma_tau = np.exp(gamma * tau)
        denominator = (gamma + self.kappa) * (exp_gamma_tau - 1) + 2 * gamma
        B = (2 * (exp_gamma_tau - 1)) / denominator
        numerator_A = 2 * gamma * np.exp((self.kappa + gamma) * tau / 2)
        A = (numerator_A / denominator) ** ((2 * self.kappa * self.theta) / self.sigma**2)
        return A, B

    def predict_yields(self, r_t, tenors_years):
        predicted_yields = []
        for tau in tenors_years:
            A, B = self._compute_A_B(tau)
            y = (B * r_t - np.log(A)) / tau
            predicted_yields.append(y)
        return np.array(predicted_yields)

# ==============================================================================
# 3. RiskNeutralCIRCalibrator Class
# ==============================================================================
class RiskNeutralCIRCalibrator:
    def __init__(self, tenor_map: Dict[str, float]):
        self.tenor_map = tenor_map
        self.kappa_q = None
        self.theta_q = None
        self.sigma_q = None

    def _compute_yield(self, r_t: float, tau: float, kappa: float, theta: float, sigma: float) -> float:
        gamma = np.sqrt(kappa**2 + 2 * sigma**2)
        exp_gamma_tau = np.exp(gamma * tau)
        denominator = (gamma + kappa) * (exp_gamma_tau - 1) + 2 * gamma
        B = (2 * (exp_gamma_tau - 1)) / denominator
        numerator_A = 2 * gamma * np.exp((kappa + gamma) * tau / 2)
        A_base = np.maximum((numerator_A / denominator), 1e-10)
        A = A_base ** ((2 * kappa * theta) / sigma**2)
        return (B * r_t - np.log(A)) / tau

    def calibrate_cross_sectional(self, train_df: pd.DataFrame) -> Tuple[float, float, float]:
        available_tenors = [col for col in train_df.columns if col in self.tenor_map]
        tenors_years = np.array([self.tenor_map[col] for col in available_tenors])

        r_t_array = train_df['3M'].values
        actual_yields = train_df[available_tenors].values

        def curve_mse_loss(params: np.ndarray) -> float:
            k, th, sig = params
            total_mse = 0
            for i, r_t in enumerate(r_t_array):
                preds = self._compute_yield(r_t, tenors_years, k, th, sig)
                total_mse += np.mean((preds - actual_yields[i]) ** 2)
            return total_mse / len(r_t_array)

        initial_guess = [0.5, 0.03, 0.05]
        bounds = ((1e-4, 5.0), (1e-4, 1.0), (1e-4, 1.0))

        print("Running Cross-Sectional (Risk-Neutral) Optimization...")
        result = minimize(curve_mse_loss, initial_guess, bounds=bounds, method='L-BFGS-B')

        self.kappa_q, self.theta_q, self.sigma_q = result.x

        print("\n--- Risk-Neutral Calibration Successful ---")
        print(f"Risk-Neutral Mean Reversion (kappa*): {self.kappa_q:.4f}")
        print(f"Risk-Neutral Long-Run Mean (theta*):  {self.theta_q:.4f}")
        print(f"Risk-Neutral Volatility (sigma*):     {self.sigma_q:.4f}")

        return self.kappa_q, self.theta_q, self.sigma_q


# ==============================================================================
# 4. CIREvaluator Class for Kalman Calibrator
# ==============================================================================

class CrossSectionalKalmanCalibrator:
    def __init__(self, tenor_map, dt=1/252):
        self.tenor_map = tenor_map
        self.dt = dt
        self.kappa_kf = None
        self.theta_kf = None
        self.sigma_kf = None

    def _compute_A_B(self, tau, k, th, sig):
        gamma = np.sqrt(k**2 + 2 * sig**2)
        exp_gamma_tau = np.exp(gamma * tau)
        denominator = (gamma + k) * (exp_gamma_tau - 1) + 2 * gamma
        B = (2 * (exp_gamma_tau - 1)) / denominator
        numerator_A = 2 * gamma * np.exp((k + gamma) * tau / 2)
        A = np.maximum((numerator_A / denominator), 1e-15) ** ((2 * k * th) / sig**2)
        return A, B

    def calibrate_panel_kalman(self, train_df):
        print("Running Cross-Sectional Kalman Filter (Panel Data)...")
        available_tenors = [col for col in train_df.columns if col in self.tenor_map]
        tau_array = np.array([self.tenor_map[col] for col in available_tenors])

        # Z is now a matrix of the ENTIRE yield curve, not just the 3M rate
        Z = train_df[available_tenors].values

        def kalman_log_likelihood(params):
            k, th, sig, R_noise = params
            log_lik = 0.0

            # Pre-compute Measurement Jacobian (H) and A_term
            H = np.zeros(len(tau_array))
            A_term = np.zeros(len(tau_array))
            for j, tau in enumerate(tau_array):
                A_val, B_val = self._compute_A_B(tau, k, th, sig)
                H[j] = B_val / tau
                A_term[j] = -np.log(A_val) / tau

            H = H.reshape(-1, 1) # Reshape to column vector

            # Initial state
            x_hat = Z[0, 0] # Start with 3M rate as guess
            P = 1e-4
            R_mat = np.eye(len(tau_array)) * (R_noise ** 2)

            for i in range(1, len(Z)):
                # 1. Predict
                x_pred = x_hat + k * (th - x_hat) * self.dt
                x_pred = max(x_pred, 1e-8)

                Q = (sig**2) * x_pred * self.dt
                P_pred = P * ((1 - k * self.dt)**2) + Q

                # 2. Update against the ENTIRE observed curve
                z_actual = Z[i].reshape(-1, 1)
                z_pred = H * x_pred + A_term.reshape(-1, 1)

                y_err = z_actual - z_pred # Innovation
                S = H @ np.array([[P_pred]]) @ H.T + R_mat

                try:
                    S_inv = np.linalg.inv(S)
                    log_lik -= 0.5 * (np.log(np.linalg.det(S)) + y_err.T @ S_inv @ y_err).item()

                    K = P_pred * H.T @ S_inv
                    x_hat = x_pred + (K @ y_err).item()
                    x_hat = max(x_hat, 1e-8)
                    P = (1 - (K @ H).item()) * P_pred
                except np.linalg.LinAlgError:
                    return 1e9 # Penalize unstable matrices

            return -log_lik

        # [kappa, theta, sigma, Measurement Noise]
        init_guess = [0.1, 0.05, 0.05, 0.01]
        bounds = ((1e-4, 5.0), (1e-4, 1.0), (1e-4, 1.0), (1e-6, 0.1))

        res = minimize(kalman_log_likelihood, init_guess, bounds=bounds, method='L-BFGS-B')

        self.kappa_kf, self.theta_kf, self.sigma_kf, R_opt = res.x

        print("\n--- Cross-Sectional Kalman Successful ---")
        print(f"Panel Kappa: {self.kappa_kf:.4f}")
        print(f"Panel Theta: {self.theta_kf:.4f}")
        print(f"Panel Sigma: {self.sigma_kf:.4f}")
        return self.kappa_kf, self.theta_kf, self.sigma_kf

# ==============================================================================
# 5. CIREvaluator Class
# ==============================================================================
class CIREvaluator:
    def __init__(self, builder: CIRYieldCurveBuilder):
        """
        Initializes the evaluator with our prediction builder.
        """
        self.builder = builder

        # Define the mathematical map of all possible target tenors
        self.maturity_map = {
            '6M': 0.5, '9M': 0.75, '1Y': 1.0, '2Y': 2.0,
            '5Y': 5.0, '10Y': 10.0, '20Y': 20.0, '30Y': 30.0
        }

    def evaluate(self, test_df: pd.DataFrame):
        """
        Dynamically detects available maturities, reconstructs the yield curves,
        and calculates the R-squared score.
        """
        # 1. Dynamically detect which target columns exist in the provided dataset
        available_targets = [col for col in test_df.columns if col in self.maturity_map]
        tenors_years = [self.maturity_map[col] for col in available_targets]

        print(f"--- Dataset Alignment ---")
        print(f"Detected {len(available_targets)} target maturities in test data: {available_targets}")

        # 2. Extract features and actuals
        r_t_series = test_df['3M'].values
        actuals = test_df[available_targets].values

        # 3. Generate predictions ONLY for the available tenors
        predictions = []
        for r_t in r_t_series:
            pred = self.builder.predict_yields(r_t, tenors_years)
            predictions.append(pred)

        predictions = np.array(predictions)

        # 4. Calculate Global R2 score
        global_r2 = r2_score(actuals.flatten(), predictions.flatten())

        # Calculate per-tenor R2
        per_tenor_r2 = r2_score(actuals, predictions, multioutput='raw_values')

        print(f"\n--- Out-of-Sample Evaluation ---")
        print(f"Global R-squared Score: {global_r2:.4f}")

        if global_r2 > 0.85:
            print("✅ PASSED: Score is greater than the required 0.85 threshold.")
        else:
            print(" FAILED: Score is below the 0.85 threshold. We will need our extension to fix this.")

        print("\n--- R-squared per Tenor ---")
        for label, r2 in zip(available_targets, per_tenor_r2):
            print(f"{label:<5}: {r2:.4f}")

        return global_r2, predictions, actuals

# ==============================================================================
# 5. Unified Execution Block (MLE vs. Kalman Filter)
# ==============================================================================
print("--- Preparing Data ---")
master_tenor_map = {
    '6M': 0.5, '9M': 0.75, '1Y': 1.0, '2Y': 2.0,
    '5Y': 5.0, '10Y': 10.0, '20Y': 20.0, '30Y': 30.0
}

preprocessor_train = YieldCurvePreprocessor('train_data.csv')
train_df = preprocessor_train.process()

preprocessor_test = YieldCurvePreprocessor('test_data.csv')
cleaned_test_df = preprocessor_test.process()


# ------------------------------------------------------------------------------
# MODEL 1: CROSS-SECTIONAL (MLE / RISK-NEUTRAL) EVALUATION
# ------------------------------------------------------------------------------
print("\n==========================================================")
print(" MODEL 1: MLE (CROSS-SECTIONAL) CALIBRATION")
print("==========================================================")
rn_calibrator = RiskNeutralCIRCalibrator(tenor_map=master_tenor_map)
kappa_q, theta_q, sigma_q = rn_calibrator.calibrate_cross_sectional(train_df)

rn_builder = CIRYieldCurveBuilder(kappa=kappa_q, theta=theta_q, sigma=sigma_q)
rn_evaluator = CIREvaluator(builder=rn_builder)

print("\n>>> Running MLE Out-of-Sample Evaluation...")
rn_global_r2, rn_test_predictions, test_actuals = rn_evaluator.evaluate(cleaned_test_df)


# ------------------------------------------------------------------------------
# MODEL 2: CROSS-SECTIONAL KALMAN FILTER EVALUATION
# ------------------------------------------------------------------------------
print("\n==========================================================")
print(" MODEL 2: KALMAN FILTER (CROSS-SECTIONAL) CALIBRATION")
print("==========================================================")

cs_kalman_calibrator = CrossSectionalKalmanCalibrator(tenor_map=master_tenor_map)
kappa_kf, theta_kf, sigma_kf = cs_kalman_calibrator.calibrate_panel_kalman(train_df)

kf_builder = CIRYieldCurveBuilder(kappa=kappa_kf, theta=theta_kf, sigma=sigma_kf)
kf_evaluator = CIREvaluator(builder=kf_builder)

print("\n>>> Running Panel Kalman Out-of-Sample Evaluation...")
kf_global_r2, kf_test_predictions, _ = kf_evaluator.evaluate(cleaned_test_df)

# ------------------------------------------------------------------------------
# 6. FINAL SIDE-BY-SIDE COMPARISON
# ------------------------------------------------------------------------------
print("\n==========================================================")
print(" FINAL OOS COMPARISON: MLE vs KALMAN FILTER")
print("==========================================================")
print(f"Global R^2 (MLE / Cross-Sectional): {rn_global_r2:.4f}")
print(f"Global R^2 (Kalman / Time-Series):  {kf_global_r2:.4f}")
print("-" * 65)

# Calculate per-tenor R2 for side-by-side display
available_targets = [col for col in cleaned_test_df.columns if col in master_tenor_map]
rn_per_tenor_r2 = r2_score(test_actuals, rn_test_predictions, multioutput='raw_values')
kf_per_tenor_r2 = r2_score(test_actuals, kf_test_predictions, multioutput='raw_values')

print(f"{'Tenor':<10} | {'MLE R-squared':<22} | {'Kalman R-squared':<22}")
print("-" * 65)
for label, rn_r2, kf_r2 in zip(available_targets, rn_per_tenor_r2, kf_per_tenor_r2):
    print(f"{label:<10} | {rn_r2:<22.4f} | {kf_r2:<22.4f}")

## Here we got the Global R2 score by flattening the data on Baase CIR Model
### Global R-squared Score using MLE Calibration is 0.8922
and
### Global R-squared Score using Kalman Filter is 0.8762

### Evaluation Criteria: Out-of-Sample Prediction Performance

A core requirement of this project is achieving an out-of-sample $R^2$ score greater than **0.85** on the test dataset when reconstructing the yield curve from the 3M rate.

As demonstrated in our previous time-series analysis, a naive physical calibration of the single-factor CIR model systematically fails to fit the broader curve. To overcome this and achieve our passing scores of **0.8922** (MLE) and **0.8762** (Kalman), we adapted our methodology from a strict time-series approach to a Cross-Sectional (Panel Data) Calibration, effectively shifting from the physical measure ($\mathbb{P}$) to the risk-neutral measure ($\mathbb{Q}$).

#### Adaptations to the Base CIR Model

The standard SDE models the "real world" physical evolution of the short rate. However, bond prices in the market embed a "market price of risk." If we ignore this risk premium, our curve predictions will systematically deviate from actual market prices.

To correct this without immediately abandoning the single-factor framework, we altered the objective functions of our calibrators:

* **Risk-Neutral Cross-Sectional MLE:** Instead of just maximizing the likelihood of the 3M rate's day-to-day transitions, our `curve_mse_loss` function minimizes the Mean Squared Error across all available tenors on the curve simultaneously. This forces the optimizer to find the risk-adjusted parameters ($\kappa^*, \theta^*, \sigma^*$) that best explain the shape of the curve, rather than just the history of the short rate.
* **Panel Kalman Filter:** We expanded the measurement matrix ($Z$) in the update step from a 1D observation (just the 3M rate) to an N-dimensional observation (the entire yield curve). By penalizing the filter for cross-sectional prediction errors, the hidden state estimates become heavily weighted by the broader term structure.

#### Key Observations

* **The "Short-End" Bias:** The models perform phenomenally well on the short end of the curve (6M and 9M), achieving near-perfect $R^2$ scores **> 0.96**. Because these tenors are highly correlated with the 3M input rate, the affine $A(t,T)$ and $B(t,T)$ transformations map them almost flawlessly.
* **The "Belly" Decay:** Performance decays dramatically as we step further out on the curve. By the 2Y tenor, the $R^2$ collapses to **~0.38**.
* **The Global Score Illusion:** Why is the global score so high (**> 0.85**) if the 2Y is failing? The test dataset provided for this specific evaluation chunk only contained short-to-medium tenors (6M, 9M, 1Y, 2Y). Because 3 out of the 4 targets are structurally bound to the short rate, the global average is dragged upward. If the 10Y, 20Y, or 30Y tenors were present in this specific test set, the global score would likely plunge below the **0.85** threshold.

---

### Conclusion

By transitioning to a risk-neutral, cross-sectional calibration, we mathematically squeezed the absolute maximum predictive power out of the Base CIR model, officially passing the project's baseline accuracy requirement. However, the severe $R^2$ decay at the 2-Year mark proves that a single stochastic factor cannot explain the independent movements of medium and long-term bonds. This perfectly sets the stage for implementing advanced extensions—like a Two-Factor model—to fix the mid-curve collapse.

In [ ]:
# ==============================================================================
# CIR++ Class Definition
# ==============================================================================
class CIRPlusPlusBuilder(CIRYieldCurveBuilder):
    def __init__(self, kappa, theta, sigma):
        super().__init__(kappa, theta, sigma)
        self.shifts = {}

    def calibrate_shifts(self, train_df: pd.DataFrame, tenor_map: dict, model_name=""):
        print(f"\n--- Calibrating CIR++ Shifts for {model_name} ---")
        available_tenors = [col for col in train_df.columns if col in tenor_map]
        tenors_years = [tenor_map[col] for col in available_tenors]

        r_t_series = train_df['3M'].values
        actuals = train_df[available_tenors].values

        base_predictions = []
        for r_t in r_t_series:
            base_predictions.append(super().predict_yields(r_t, tenors_years))
        base_predictions = np.array(base_predictions)

        residuals = actuals - base_predictions
        mean_shifts = np.mean(residuals, axis=0)

        for label, shift in zip(available_tenors, mean_shifts):
            self.shifts[label] = shift
            print(f"Shift phi({label:<2}): {shift:+.6f} (Decimal)")

    def predict_yields_plus(self, r_t: float, target_tenors: list, tenor_map: dict) -> np.ndarray:
        tenors_years = [tenor_map[col] for col in target_tenors]
        base_yields = super().predict_yields(r_t, tenors_years)

        shifted_yields = []
        for i, label in enumerate(target_tenors):
            shift_val = self.shifts.get(label, 0.0)
            shifted_yields.append(base_yields[i] + shift_val)

        return np.array(shifted_yields)

# ==============================================================================
# Unified CIR++ Execution Block (MLE & Kalman)
# ==============================================================================
print("--- Implementing CIR++ Extension for Both Models ---")

# 1. Instantiate BOTH CIR++ Builders
mle_plus_builder = CIRPlusPlusBuilder(kappa=kappa_q, theta=theta_q, sigma=sigma_q)
kf_plus_builder = CIRPlusPlusBuilder(kappa=kappa_kf, theta=theta_kf, sigma=sigma_kf)

# 2. Calibrate deterministic shifts for BOTH models using Training Data
mle_plus_builder.calibrate_shifts(train_df, master_tenor_map, model_name="MLE")
kf_plus_builder.calibrate_shifts(train_df, master_tenor_map, model_name="Kalman Filter")

# 3. Generate Out-of-Sample Predictions for BOTH models
available_targets = [col for col in cleaned_test_df.columns if col in master_tenor_map]
r_t_test_series = cleaned_test_df['3M'].values
actuals_test = cleaned_test_df[available_targets].values

mle_predictions_plus = []
kf_predictions_plus = []

for r_t in r_t_test_series:
    mle_predictions_plus.append(mle_plus_builder.predict_yields_plus(r_t, available_targets, master_tenor_map))
    kf_predictions_plus.append(kf_plus_builder.predict_yields_plus(r_t, available_targets, master_tenor_map))

mle_predictions_plus = np.array(mle_predictions_plus)
kf_predictions_plus = np.array(kf_predictions_plus)

# 4. Calculate Final CIR++ R2 Scores
mle_global_r2_plus = r2_score(actuals_test.flatten(), mle_predictions_plus.flatten())
kf_global_r2_plus = r2_score(actuals_test.flatten(), kf_predictions_plus.flatten())

mle_per_tenor_r2_plus = r2_score(actuals_test, mle_predictions_plus, multioutput='raw_values')
kf_per_tenor_r2_plus = r2_score(actuals_test, kf_predictions_plus, multioutput='raw_values')

# 5. Output Master Comparison Table
print("\n==================================================================")
print(" FINAL OOS COMPARISON: BASE MODELS vs CIR++ EXTENSIONS")
print("==================================================================")
print(f"MLE Base Global R^2:    {rn_global_r2:.4f}  |  MLE CIR++ Global R^2:    {mle_global_r2_plus:.4f}")
print(f"Kalman Base Global R^2: {kf_global_r2:.4f}  |  Kalman CIR++ Global R^2: {kf_global_r2_plus:.4f}")
print("-" * 66)
print(f"{'Tenor':<8} | {'MLE CIR++ R^2':<20} | {'Kalman CIR++ R^2':<20}")
print("-" * 66)
for label, mle_r2, kf_r2 in zip(available_targets, mle_per_tenor_r2_plus, kf_per_tenor_r2_plus):
    print(f"{label:<8} | {mle_r2:<20.4f} | {kf_r2:<20.4f}")

### Model Improvement & Extensions: The CIR++ Paradox

To address the structural limitations of the single-factor CIR framework, we implemented the CIR++ extension. As developed by Brigo and Mercurio, this extension introduces deterministic, time-dependent parameters—specifically a shift function $\phi(t)$—to allow the model to exactly fit the initial term structure.  

In our implementation, we calculated this shift by finding the mean residual error for each tenor across the training dataset and applying it as a static structural premium to our out-of-sample predictions.

#### The Paradoxical Result: Decreased $R^2$

Counterintuitively, applying the CIR++ extension resulted in a lower out-of-sample $R^2$ score compared to the Base Cross-Sectional models.

This directly answers a key project question: *"Does your extension meaningfully improve out-of-sample performance, or does it overfit the training period?"* The CIR++ model severely overfit the training period.  

#### Why Does This Happen? (The Mechanics of Overfitting)

* **Memorizing Historical Term Premia:** The base CIR model predicts the "pure" mathematical yield based on mean reversion. The residuals (the difference between actual yields and base predictions) represent the market's specific risk premium or structural bias during the training period. By calculating $\phi$ from the training data, the CIR++ model essentially "memorized" the exact shape and pricing errors of that specific historical regime.
* **Vulnerability to Regime Shifts:** Financial markets are non-stationary. The term premium that existed in the training data (e.g., a steep, upward-sloping curve) might completely reverse in the test data (e.g., a flat or inverted curve due to an economic shock).
    * When we project the **base CIR model** forward, it adapts dynamically based on the new 3M rate.
    * When we project the **CIR++ model** forward, it takes that dynamic prediction and stubbornly adds the $\phi$ shift from the past regime. If the market dynamics have changed, this deterministic shift pushes the prediction in the exact wrong direction, compounding the error and destroying the $R^2$ score.

#### The Danger of Deterministic Fixes in Stochastic Environments

This highlights a fundamental theoretical limitation of the CIR++ approach in practical trading scenarios. While it is mathematically elegant for exactly pricing derivatives on "Day 1" (because it fits today's curve perfectly), its predictive power decays rapidly over time. It assumes the pricing errors of the past will remain constant in the future, which violates the core reality of market dynamics.

---

### Conclusion

The failure of the basic CIR++ shift to improve out-of-sample prediction validates our earlier cross-sectional calibration. It proves that to genuinely improve the model, we cannot simply "patch" the errors with static historical averages.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

def plot_regime_shift_failure_interactive(test_df, base_predictions, plus_predictions, target_tenors):
    """
    Creates an interactive slider to visualize the Actual vs Base CIR vs CIR++ curves
    for every day in the out-of-sample test set.
    """
    # Map tenors to numerical x-axis values for plotting
    tenor_to_years = {'6M': 0.5, '9M': 0.75, '1Y': 1.0, '2Y': 2.0, '5Y': 5.0, '10Y': 10.0, '20Y': 20.0, '30Y': 30.0}
    x_axis = [tenor_to_years[t] for t in target_tenors]

    # Pre-calculate global min/max so the y-axis doesn't bounce around while scrolling
    global_min = min(test_df[target_tenors].min().min(), np.min(base_predictions), np.min(plus_predictions))
    global_max = max(test_df[target_tenors].max().max(), np.max(base_predictions), np.max(plus_predictions))
    y_margin = (global_max - global_min) * 0.1

    def update_plot(day_index):
        sample_date = test_df.index[day_index]
        actual_curve = test_df[target_tenors].iloc[day_index].values
        base_curve = base_predictions[day_index]
        plus_curve = plus_predictions[day_index]

        plt.figure(figsize=(10, 6))
        plt.plot(x_axis, actual_curve, 'k-o', linewidth=2.5, label=f'Actual Market Curve')
        plt.plot(x_axis, base_curve, 'b--s', linewidth=2, label='Base CIR (Risk-Neutral)')
        plt.plot(x_axis, plus_curve, 'r--^', linewidth=2, label='CIR++ (Overfitted Shift)')

        plt.title(f"Yield Curve Reconstruction on {sample_date.date()}", fontsize=14, fontweight='bold')
        plt.xlabel("Maturity (Years)", fontsize=12)
        plt.ylabel("Yield (Decimal)", fontsize=12)
        plt.xticks(x_axis, target_tenors)

        # Lock the y-axis for smooth visual comparison across time
        plt.ylim(global_min - y_margin, global_max + y_margin)

        plt.legend(loc='upper right', fontsize=11)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()

    # Create the interactive slider
    total_days = len(test_df)
    slider = widgets.IntSlider(
        value=total_days - 25, # Default starting position
        min=0,
        max=total_days - 1,
        step=1,
        description='Scrub Days:',
        layout=widgets.Layout(width='80%'),
        continuous_update=False # Set to True if your Colab environment is fast enough to render in real-time
    )

    print("--- Interactive Regime Shift Analysis ---")
    print("Use the slider below to drag through every date in the test period.")
    widgets.interact(update_plot, day_index=slider)

# --- Execution Block ---
plot_regime_shift_failure_interactive(
    cleaned_test_df,
    rn_test_predictions, # From your Base model block
    mle_predictions_plus, # From your CIR++ block
    available_targets
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from mpl_toolkits.mplot3d import Axes3D

def plot_advanced_diagnostics(test_df, predictions, actuals, targets, tenor_map):
    """
    Generates a Residual Heatmap and a 3D Surface Plot for the out-of-sample predictions.
    """
    # 1. Calculate Absolute Residuals (Errors)
    residuals = np.abs(actuals - predictions)
    residual_df = pd.DataFrame(residuals, index=test_df.index, columns=targets)

    # ==========================================
    # PLOT 1: Residual Heatmap
    # ==========================================
    plt.figure(figsize=(12, 6))
    # Transpose so tenors are on the Y-axis and Time is on the X-axis
    sns.heatmap(residual_df.T, cmap="YlOrRd", robust=True,
                cbar_kws={'label': 'Absolute Error (Decimal Yield)'})

    plt.title("Out-of-Sample Prediction Error Heatmap", fontsize=16, fontweight='bold')
    plt.xlabel("Test Set Timeline", fontsize=12)
    plt.ylabel("Maturity Tenor", fontsize=12)

    # Format the x-axis dates nicely (showing roughly 10 ticks to avoid crowding)
    xticks = np.linspace(0, len(residual_df.index) - 1, 10, dtype=int)
    xtick_labels = [residual_df.index[i].strftime('%Y-%m-%d') for i in xticks]
    plt.xticks(ticks=xticks, labels=xtick_labels, rotation=45)

    plt.tight_layout()
    plt.show()

    # ==========================================
    # PLOT 2: 3D Yield Curve Surface
    # ==========================================
    fig = plt.figure(figsize=(14, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Create the meshgrid for X (Time) and Y (Maturity in Years)
    numeric_dates = mdates.date2num(test_df.index)
    tenors_in_years = [tenor_map[t] for t in targets]

    X, Y = np.meshgrid(numeric_dates, tenors_in_years)
    # Z-axis is the predicted yield (needs to be transposed to match meshgrid dimensions)
    Z = predictions.T

    # Plot the surface
    surf = ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none', alpha=0.9)

    ax.set_title("3D Spatio-Temporal Evolution of the Predicted Yield Curve", fontsize=16, fontweight='bold')
    ax.set_ylabel("Maturity (Years)", fontsize=12, labelpad=10)
    ax.set_zlabel("Predicted Yield", fontsize=12, labelpad=10)

    # Format the X-axis (Time) to show dates instead of raw floats
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    # Rotate date labels for readability
    ax.tick_params(axis='x', rotation=45)

    # Add a color bar
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1, label='Yield Level')

    plt.tight_layout()
    plt.show()

# --- Execution Block ---
# Ensure this runs directly after your Risk-Neutral evaluation block
# so 'cleaned_test_df', 'test_predictions', and 'test_actuals' are populated by the MLE model.

print("--- Generating Advanced Financial Diagnostics ---")
plot_advanced_diagnostics(
    cleaned_test_df,
    rn_test_predictions,
    test_actuals,
    available_targets,
    master_tenor_map
)

# **Critical Analysis: Visualizing Model Mechanics and Failure Regimes**

To fully articulate the practical and theoretical limitations of our calibrated Base CIR model, we generated advanced spatio-temporal diagnostics on the out-of-sample test data. These visual tools expose the precise scenarios where the single-factor assumption fails when confronted with real market dynamics.

#### 1. The 3D Spatio-Temporal Surface Plot (Structural Rigidity)

The 3D surface plot visualizes the predicted yield curve across both time (x-axis) and maturity (y-axis).

* **The Single-Factor "Lockstep":** Looking at the surface, you can see that the entire yield curve moves in near-perfect lockstep. When the short end drops, the long end smoothly follows.
* **Theoretical Limitation:** This perfectly illustrates the mathematical rigidity of a single-factor affine model. Because the entire 3D surface is driven by just one stochastic variable (the 3M rate, $r_t$), the model cannot generate independent "ripples", "humps", or inversions in the middle of the curve. It forces a deterministic, highly correlated shape across all maturities, which contradicts empirical market behavior where different segments of the curve are driven by different macroeconomic forces.

#### 2. The Out-of-Sample Error Heatmap (Time-Dependent Shock Failures)

The heatmap provides a forensic breakdown of our absolute prediction errors (residuals) across both tenors and the timeline. Darker red indicates severe mispricing (errors exceeding **~0.006**, or **60 basis points**), while pale yellow indicates high accuracy.

* **The Cross-Sectional Breakdown:** Confirming our $R^2$ analysis, the top rows (6M and 9M) are a solid, pale yellow across the entire two-year test period. The model handles these maturities flawlessly. However, the bottom row (2Y) is highly volatile.
* **Macroeconomic Shock Regimes:** Notice that the massive 2Y errors are not evenly distributed; they cluster into violent, dark red vertical bands (specifically visible around Q3/Q4 2024 and early 2026).
* **The "Why":** These vertical bands represent periods of market stress or shifting central bank forward guidance. During these specific windows, traders aggressively bought or sold 2-Year bonds based on future interest rate expectations, completely detaching the 2Y yield from the current instantaneous 3M rate. Our model, utterly blind to forward expectations and relying solely on the grounded 3M rate, suffered catastrophic prediction failures.

#### Implications for Real-World Trading and Risk Management

In a real-world scenario, relying on this single-factor risk-neutral model would be highly dangerous for portfolio management.

If a trading desk used this model to delta-hedge a portfolio of 2-Year bonds using 3-Month derivatives, they would assume their risk was neutralized. However, as the heatmap shows, during the regime shocks of late 2024, the 2Y bond prices deviated massively from the model's predictions (by over **60 basis points**). The desk would suddenly find themselves holding a massive, unhedged mark-to-market loss.

---

### Final Conclusion

While the cross-sectionally calibrated CIR model is mathematically elegant and passes the project's baseline accuracy checks due to its dominance on the short end, it is practically unusable for managing mid-to-long-term duration risk. True market dynamics require multi-factor models capable of capturing both the current rate (level) and the independent shifting expectations of the future (slope and curvature).